# Assignment 2: Counter-Narrative Generation (worked solution, version 7, T4 GPU)

<i><u>**Study aid – not part of the core workbook.** Skip this part when you write your own submission.</u></i>

## <i><u>About this notebook</u></i>

<i><u>This notebook is a complete, heavily commented solution to **CT5146 Assignment 2**, written to be **studied**: every choice is explained and linked back to the course materials.</u></i>

- <i><u>**Text in italics and underlined marks a study aid** (like this section). Study aids are **not part of the core workbook**: skip them when you write your own submission.</u></i>
- <i><u>The answers to 1c, 2c and 3c come in three versions: **A** is the short core answer (max 50 words); **B** (long explanation) and **C** (worked example with the real results of the 100-item run, 22 October 2026) are study aids.</u></i>
- <i><u>Running this notebook should reproduce the same numbers, because generation and judging use greedy decoding.</u></i>

## <i><u>How to run it in Google Colab</u></i>

1. <i><u>**Runtime → Change runtime type → T4 GPU.**</u></i>
2. <i><u>**Runtime → Run all.**</u></i>
3. <i><u>**Expected runtime: about 35–45 minutes** with `N_TEST = 100` (measured in the 100-item run):</u></i>
   - <i><u>downloading the models takes roughly 10–15 minutes;</u></i>
   - <i><u>generating 100 replies takes about 4–5 minutes per method;</u></i>
   - <i><u>judging 100 replies takes about 7 minutes per method.</u></i>

   <i><u>For a quick first test, set `N_TEST = 20`.</u></i>
4. <i><u>Colab deletes files when the session ends. Download the generated CSV files from the **Files** panel if you want to keep them.</u></i>
5. <i><u>**When you are finished, choose Runtime → Disconnect and delete runtime**, so an idle session does not keep using your GPU time.</u></i>

## <i><u>How references to the course are written</u></i>

| <i><u>Short name</u></i> | <i><u>Course file</u></i> |
|---|---|
| <i><u>**W2**</u></i> | <i><u>`CT5146_Week2.pdf` (linguistic concepts)</u></i> |
| <i><u>**L3**</u></i> | <i><u>`CT5146-3.pdf` (vector space models, word embeddings)</u></i> |
| <i><u>**L4**</u></i> | <i><u>`4 Text Classification.pdf`</u></i> |
| <i><u>**L5**</u></i> | <i><u>`5 Language Models.pdf`</u></i> |
| <i><u>**L6**</u></i> | <i><u>`ct5146-6.pdf` (6a LLMs, 6b Prompting)</u></i> |
| <i><u>**L7**</u></i> | <i><u>`7 RAG and Generation.pdf` (7a RAG, 7b Generation)</u></i> |
| <i><u>**Lab7**</u></i> | <i><u>`07_transformer_Prompting_solution_2025_2026.ipynb`</u></i> |
| <i><u>**WS6b / WS7b**</u></i> | <i><u>Worksheets "6b Prompting" and "7b Generation"</u></i> |
| <i><u>**SK**</u></i> | <i><u>`starter_kit_counter_narrative.ipynb` (shared by the tutor)</u></i> |
| <i><u>**Tutor**</u></i> | <i><u>Assignment Q&A session with the tutor</u></i> |
| <i><u>**Peers**</u></i> | <i><u>Class chat (students relaying Q&A answers)</u></i> |

<i><u>For example, "L7 p.44" means page 44 of the RAG and Generation slides.</u></i>

> <i><u>**Note on the written answers (1c, 2c, 3c):**</u></i>
> - <i><u>The brief says they must be "written in your own words", so use the versions below to study and rewrite them yourself before submitting anything.</u></i>
> - <i><u>Version C uses the **real results of the 100-item run** (22 October 2026).</u></i>

## Task roadmap (from the brief)

| # | Section | Focus | Marks |
|---|---|---|---:|
| 1 | Dataset & Task Understanding | Explore the hate-speech / counter-narrative dataset | 30 |
| 2 | Counter-Narrative Generation Method | Instruction-based LLM generation + evaluation | 40 |
| 3 | Improving Generation & Comparison | One improvement, compared with Section 2 | 30 |

<i><u>**Study aid – not part of the core workbook.** Skip this part when you write your own submission.</u></i>

## <i><u>All design choices at a glance</u></i>

<i><u>Each choice is explained in detail in its section.</u></i>

| <i><u>Decision</u></i> | <i><u>Choice</u></i> | <i><u>Main reason (course source)</u></i> |
|---|---|---|
| <i><u>Hardware</u></i> | <i><u>**T4 GPU** (15 GB usable)</u></i> | <i><u>Colab's standard GPU; room for a 3B writer and a 7B judge in 4-bit; uses `float16` (no `bfloat16` on the T4)</u></i> |
| <i><u>Dataset</u></i> | <i><u>**Multitarget-CONAN**</u></i> | <i><u>Exactly the 3 required columns, 8 target groups, English, expert-written</u></i> |
| <i><u>Train/test split</u></i> | <i><u>`train_test_split` on the **unique hate speeches** (80/20, stratified by target)</u></i> | <i><u>No hate speech in both parts, so retrieval cannot "cheat"; the same target mix in both parts</u></i> |
| <i><u>Test subset</u></i> | <i><u>**100** hate speeches</u></i> | <i><u>The brief says "test subset"; full runs took classmates over 1.5 h</u></i> |
| <i><u>Generation LLM</u></i> | <i><u>**Qwen/Qwen2.5-3B-Instruct**, 4-bit</u></i> | <i><u>Open-source, instruction-tuned (L6 p.43), decoder-only (L3 p.91); newer and larger than the lab models, so expected to follow the rules better; fits the T4</u></i> |
| <i><u>Quantisation</u></i> | <i><u>4-bit NF4, the **Lab7 configuration**</u></i> | <i><u>Both models fit in GPU memory with only a small quality loss</u></i> |
| <i><u>Library</u></i> | <i><u>Hugging Face `transformers` pipeline</u></i> | <i><u>The same as Lab7; Ollama is allowed (Tutor) but awkward in Colab</u></i> |
| <i><u>Prompt</u></i> | <i><u>Starter-kit structure (Task / Rules / Comment) + persona and rules from AUTOMAT</u></i> | <i><u>SK; L6 pp.46–51, 58, 65–68</u></i> |
| <i><u>Decoding</u></i> | <i><u>Greedy (`do_sample=False`), max 120 new tokens</u></i> | <i><u>Reproducible; less hallucination than sampling (L7 pp.44, 47–49; WS7b)</u></i> |
| <i><u>Reference metrics</u></i> | <i><u>BLEU, ROUGE-1/2/L, BERTScore</u></i> | <i><u>Named in the brief; BLEU code from Lab7; BERTScore measures meaning (L7 p.45)</u></i> |
| <i><u>Rubric</u></i> | <i><u>4 criteria × **strict 1–5 scale**: relevance, tone, factuality, clarity</u></i> | <i><u>Own rubric allowed (tutor, peers); L7 p.41, p.43; "fact-bound" in the brief; a first judge with a 1–3 scale gave every reply full marks</u></i> |
| <i><u>Rubric scorer</u></i> | <i><u>LLM judge **Qwen/Qwen2.5-7B-Instruct** (larger than the writer); **critique first**, then scores in **JSON**</u></i> | <i><u>"LLM can be asked to judge" (L7 p.27); chain-of-thought (L6 p.55); JSON output (L6 p.58); no hand scoring</u></i> |
| <i><u>Extra measure</u></i> | <i><u>**Refusal rate**</u></i> | <i><u>The tutor asked for it explicitly</u></i> |
| <i><u>Improvement (Section 3)</u></i> | <i><u>**Retrieval-augmented few-shot prompting**: the 3 most similar training pairs go into the prompt</u></i> | <i><u>Few-shot (L6 p.54) + Retrieve-Read RAG (L7 p.5) + dense retrieval (L7 pp.12–14)</u></i> |
| <i><u>Embedding model</u></i> | <i><u>`all-MiniLM-L6-v2` (Sentence Transformers)</u></i> | <i><u>Small and fast; sentence embeddings (L7 p.13); cosine similarity (L3 p.31)</u></i> |

<i><u>**Study aid – not part of the core workbook.** Skip this part when you write your own submission.</u></i>

## <i><u>Tiny Python refresher (for reading this notebook)</u></i>

- <i><u>**list**: `[1, 2, 3]`, an ordered collection. `my_list.append(x)` adds an item.</u></i>
- <i><u>**dict**: `{"name": "Ann", "age": 30}`, key → value pairs. Read a value with `my_dict["name"]`.</u></i>
- <i><u>**f-string**: `f"Hello {name}"` puts the value of the variable `name` into the text. Triple quotes `f"""..."""` allow several lines. In this notebook, f-strings only ever contain plain variable names.</u></i>
  - <i><u>`{average_similarity:.3f}` means "show with 3 decimal places".</u></i>
- <i><u>**function**: `def add(a, b): return a + b`, a reusable piece of code.</u></i>
- <i><u>**for loop**: `for item in my_list:` repeats the indented code once for every item.</u></i>
  - <i><u>`enumerate(my_list, start=1)` gives `(number, item)` pairs.</u></i>
  - <i><u>`zip(list_a, list_b)` walks through two lists side by side.</u></i>
- <i><u>**pandas DataFrame**: a table.</u></i>
  - <i><u>`df["COLUMN"]` is one column.</u></i>
  - <i><u>`df[condition]` keeps the rows where the condition is True.</u></i>
  - <i><u>`df.groupby("X")` makes groups of rows that share the same value in X.</u></i>
- <i><u>**`display(table)`** shows a table nicely in a notebook.</u></i>
- <i><u>**`tqdm(my_list)`** works like `my_list` in a for loop, but also shows a progress bar.</u></i>

---
# Section 0: Setup

### Installing packages

Colab already includes `torch`, `transformers`, `accelerate`, `scikit-learn`, `pandas`, `matplotlib` and `tqdm`. We only add:

| Package | Why we need it |
|---|---|
| `bitsandbytes` | 4-bit loading of the LLMs (also installed in Lab7) |
| `evaluate` | Loads BLEU / ROUGE / BERTScore with one line each (the same library as Lab7) |
| `rouge_score`, `bert_score` | Needed by `evaluate` for ROUGE and BERTScore |
| `sentence-transformers` | Sentence embeddings for retrieval in Section 3 (L7 p.13) |

In [ ]:
# The "!" at the start means: run this as a terminal command, not as Python.
# "-q" means quiet (less output).
!pip install -q bitsandbytes evaluate rouge_score bert_score sentence-transformers

In [ ]:
# ----- Standard Python module -----
import json      # to read the judge's answers, which are written in JSON format

# ----- Data analysis, plotting, progress bars, data splitting -----
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# ----- Deep learning / LLMs -----
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, set_seed

# ----- Evaluation and retrieval -----
import evaluate
from sentence_transformers import SentenceTransformer, util

# Hide long warning messages from transformers (they are harmless, see the note in Section 2a).
transformers.logging.set_verbosity_error()

# Show long texts in tables without cutting them off too early.
pd.set_option("display.max_colwidth", 200)

### Settings

All important settings are collected in one cell, so you can change them in one place.

- **`RANDOM_SEED = 42`**: every split and sample gives the same result each time (reproducible results). Any fixed number works; 42 is just a popular convention (a joke from *The Hitchhiker's Guide to the Galaxy*). Changing it would give a different split and test sample, and therefore slightly different numbers.
- **`TEST_SIZE = 0.2`**: a standard 80/20 train/test split.
- **`N_TEST = 100`**: we generate for 100 test hate speeches, not all ~740.
  - The brief itself says "on a test subset", and classmates reported over 1.5 hours for full runs.
  - 100 is a compromise: big enough for stable averages, small enough to run both methods plus the judge in about 35–45 minutes on a T4.
- **`MAX_NEW_TOKENS = 120`**: the prompt asks for at most 60 words, which is roughly 80 tokens; 120 leaves a safety margin. The starter kit (SK) also used 120.
- **`JUDGE_MAX_NEW_TOKENS = 150`**: the judge writes a one-sentence critique plus the scores in JSON (about 60–100 tokens); 150 leaves a safety margin.
- **`COMPUTE_DTYPE = torch.float16`**: the 16-bit number format the models calculate with. The T4 supports `float16` (the newer `bfloat16` needs an L4 or A100).
- **`K_EXAMPLES = 3`**: the number of retrieved examples in Section 3.
  - L6 p.54 says few-shot uses "a few (1–5) examples".
  - 3 gives variety without making the prompt too long ("overly long prompts can hurt", L6 p.49).

In [ ]:
RANDOM_SEED = 42

# Data
DATA_URL = (
    "https://raw.githubusercontent.com/marcoguerini/CONAN/master/"
    "Multitarget-CONAN/Multitarget-CONAN.csv"
)
TEST_SIZE = 0.2          # 20% of the unique hate speeches go to the test set
N_TEST = 100             # how many test hate speeches we generate for (use 20 for a quick try)

# Models (all open-source, no sign-up needed)
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"         # writes the counter-narratives
JUDGE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # scores them with the rubric (larger than the writer)
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # retrieval in Section 3

# Generation
MAX_NEW_TOKENS = 120         # maximum length of one generated counter-narrative
JUDGE_MAX_NEW_TOKENS = 150   # maximum length of one judge answer (critique + scores)
K_EXAMPLES = 3               # number of retrieved examples in Section 3
COMPUTE_DTYPE = torch.float16   # 16-bit calculations (the format the T4 supports)

# Make everything reproducible
set_seed(RANDOM_SEED)    # sets the seed for Python, NumPy and PyTorch at once

---
# Section 1: Dataset & Counter-Narrative Task Understanding (30 marks)

## Choice of dataset: Multitarget-CONAN

The brief offers four example datasets (Multitarget-CONAN is listed twice) or "another appropriate publicly available counter-narrative dataset". I chose **Multitarget-CONAN** (Fanton et al., 2021) for these reasons:

1. **It already has exactly the required columns.** `HATE_SPEECH`, `COUNTER_NARRATIVE` and `TARGET` are the column names the brief asks for, so no mapping is needed.
2. **Several target groups (8):** MUSLIMS, MIGRANTS, WOMEN, LGBT+, JEWS, POC (people of colour), DISABLED and "other".
   - Section 1 asks us to analyse the target distribution, and with several targets that analysis is meaningful.
   - The original **CONAN** dataset is about **Islamophobia only**, so its target distribution would have just one class.
3. **English only.** Open LLMs and BERTScore (`lang="en"`) work best in English. The original CONAN is multilingual (English, French, Italian) and would need filtering.
4. **High-quality references.** The counter-narratives were written and checked with experts from NGOs. This matters because the human counter-narratives are our "gold standard" for evaluation.
5. **Size: about 5,000 pairs.** That is enough for a training pool for retrieval (Section 3) and a separate test set, and still small enough for fast analysis.

**Data source.**
- The dataset is downloaded from the authors' official GitHub repository (the same data as the Google Drive link in the brief).
- The extra columns `INDEX` (row number) and `VERSION` (the data-collection round, V1–V6) are not needed and are dropped.

**Loading method.** We use `pandas.read_csv`, as in Lab7. The brief's tutorial uses Hugging Face `load_dataset("csv", ...)` (as in WS6b). Both work; pandas is simpler for analysis and plotting.

In [ ]:
# Download the dataset from the authors' official GitHub repository
df = pd.read_csv(DATA_URL)

# df.info() lists every column with its number of non-empty values and its data type,
# so it shows the size of the table and whether any values are missing.
df.info()
df.head()   # show the first 5 rows

### Cleaning

We keep the cleaning minimal on purpose. The texts are given to an LLM, which handles normal punctuation and capitalisation, so the heavy preprocessing from L3 p.70 (stopword removal, stemming, lowercasing) would only damage the text.

What we do:
- keep only the 3 needed columns;
- remove spaces at the start and end of each text;
- remove exact duplicate rows (the same hate speech *and* the same counter-narrative twice; there is 1 in this dataset).

There is no missing-value step: `df.info()` above shows 5,003 non-empty values in every column, so there are no empty cells to remove.

In [ ]:
df = df[["HATE_SPEECH", "COUNTER_NARRATIVE", "TARGET"]].copy()   # keep only the needed columns
rows_before = len(df)

for column in ["HATE_SPEECH", "COUNTER_NARRATIVE", "TARGET"]:
    df[column] = df[column].str.strip()               # remove leading/trailing spaces
df = df.drop_duplicates()                             # remove exact duplicate rows
df = df.reset_index(drop=True)                        # renumber the rows 0, 1, 2, ...

rows_after = len(df)
print("Rows before cleaning:", rows_before)
print("Rows after cleaning: ", rows_after)

### Train / test split, and why it is done "by hate speech"

The brief says: use the **training data** to build the method and the retrieval examples, and the **test data** only for final evaluation.

**Important detail: the same hate speech often appears several times, each time with a different counter-narrative.** In Multitarget-CONAN, 535 hate speeches have more than one counter-narrative; one has 55. If we split the *rows* randomly, the same hate speech could end up in both train and test. Then, in Section 3, the retrieval step would find the *identical* hate speech in the training data and copy its human answer. That is **data leakage**, and it would make the improvement look much better than it really is.

**Solution: split the list of unique hate speeches, not the rows.**
1. Make a table with one row per unique hate speech (and its target).
2. Split that table with scikit-learn's standard `train_test_split` (80% / 20%).
   - `stratify=...TARGET` keeps the same share of each target group in both parts.
   - `random_state` makes the split reproducible.
3. Every row of the full data goes to the side where its hate speech went.

In [ ]:
# One row per unique hate speech, together with its target group
unique_hate_speech = df.drop_duplicates(subset="HATE_SPEECH")[["HATE_SPEECH", "TARGET"]]

train_hate_speech, test_hate_speech = train_test_split(
    unique_hate_speech,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=unique_hate_speech["TARGET"],    # same target mix in train and test
)

# .isin(...) is True for rows whose hate speech was put in the test part
is_test_row = df["HATE_SPEECH"].isin(test_hate_speech["HATE_SPEECH"])
test_df = df[is_test_row].reset_index(drop=True)
train_df = df[~is_test_row].reset_index(drop=True)     # "~" means NOT

print("Train:", len(train_df), "rows,", len(train_hate_speech), "unique hate speeches")
print("Test: ", len(test_df), "rows,", len(test_hate_speech), "unique hate speeches")

# Safety check: this must be 0, otherwise there is leakage.
overlap = set(train_df["HATE_SPEECH"]).intersection(set(test_df["HATE_SPEECH"]))
print("Hate speeches that appear in BOTH train and test:", len(overlap))

## 1a: Dataset (explore hate-speech targets and response characteristics) (10 marks)
*Brief: load your chosen dataset, count the occurrences of each target category, and compute summary statistics (lengths) of both hate speech and counter-narratives.*

What we compute, and why:
- **Target counts and percentages**: how balanced the target groups are.
- **Lengths in words and in characters.**
  - The tutor mentioned *character length* explicitly.
  - Words are easier to interpret.
  - Word counts split on spaces, which is a simple form of tokenization: they count "tokens" in the sense of W2 p.8.
- **Counter-narratives per hate speech**: shows the "one-to-many" nature of the task.

The analysis uses the **whole dataset** because it describes the data. The train/test sizes are printed above.

In [ ]:
target_counts = df["TARGET"].value_counts()                              # count per target
target_percent = df["TARGET"].value_counts(normalize=True) * 100         # share per target in %
target_table = pd.DataFrame({"count": target_counts, "percent": target_percent.round(1)})
display(target_table)

In [ ]:
# .str.split() splits a text on spaces into a list of words; .str.len() then counts the words.
df["hate_speech_words"] = df["HATE_SPEECH"].str.split().str.len()
df["counter_narrative_words"] = df["COUNTER_NARRATIVE"].str.split().str.len()
# .str.len() directly on the text counts characters.
df["hate_speech_chars"] = df["HATE_SPEECH"].str.len()
df["counter_narrative_chars"] = df["COUNTER_NARRATIVE"].str.len()

length_columns = ["hate_speech_words", "counter_narrative_words",
                  "hate_speech_chars", "counter_narrative_chars"]
print("Summary statistics of lengths:")
display(df[length_columns].describe().round(1))

In [ ]:
replies_per_hate_speech = df.groupby("HATE_SPEECH").size()     # number of rows per hate speech
n_unique = len(replies_per_hate_speech)
n_with_several = (replies_per_hate_speech > 1).sum()
most_replies = replies_per_hate_speech.max()
print("Unique hate speeches:", n_unique)
print("Hate speeches with more than one counter-narrative:", n_with_several)
print("Most counter-narratives for one hate speech:", most_replies)

## 1b: Text analysis and visualisation (10 marks)
*Brief: create two clear, labelled plots: a bar chart (or other diagram) of hate-target distributions, and a distribution comparison of input vs. response text lengths.*

- **Figure 1** is a bar chart of the target counts.
- **Figure 2** overlaps the histograms of the word lengths of hate speech (input) and counter-narratives (response), so the two distributions can be compared directly. The character lengths are in the statistics table of 1a.

Every plot has a title, axis labels and, where needed, a legend ("clear, labelled").

In [ ]:
target_counts.plot(kind="bar", figsize=(9, 4), color="steelblue", rot=0)   # pandas draws the bars
plt.title("Figure 1: Number of hate-speech / counter-narrative pairs per target group")
plt.xlabel("Target group")
plt.ylabel("Number of pairs")
plt.tight_layout()
plt.show()

In [ ]:
# Overlapping histograms; alpha=0.6 makes the bars see-through, so both stay visible
plt.figure(figsize=(9, 4))
plt.hist(df["hate_speech_words"], bins=40, range=(0, 120), alpha=0.6, label="Hate speech (input)")
plt.hist(df["counter_narrative_words"], bins=40, range=(0, 120), alpha=0.6,
         label="Counter-narrative (response)")
plt.title("Figure 2: Input vs. response text length (words)")
plt.xlabel("Number of words")
plt.ylabel("Number of texts")
plt.legend()
plt.tight_layout()
plt.show()

## 1c: Explanations (10 marks)

**Question:** Explain your dataset selection and your data analysis process. Include: target distribution; text-length differences; one important observation from your visualisations; one challenge for counter-narrative generation. (Max **50 words**, in your own words.)

### Version A: short answer (47 words)

```
I chose Multitarget-CONAN (English, expert-written, eight targets). I counted targets and measured word/character lengths. Targets are imbalanced (Muslims 27%, disabled 4%); counter-narratives are twice as long as hate speech (25 versus 13 words). Many hate speeches have several valid replies, so generating specific, factual responses is challenging.
```

<i><u>**Study aid – not part of the core workbook.** Skip this part when you write your own submission.</u></i>

### <i><u>Version B: long explanation (study notes)</u></i>

<i><u>I selected the Multitarget-CONAN dataset because it already contains the three fields the assignment requires: the hate speech, the human-written counter-narrative and the target group. No column mapping was needed. The dataset is in English, which suits open-source language models and the English model used by BERTScore. Its counter-narratives were written and reviewed together with experts from non-governmental organisations, so they are trustworthy reference answers for evaluation. Unlike the original CONAN dataset, which only covers Islamophobia, it covers eight target groups, so the target distribution can actually be analysed.</u></i>

<i><u>My analysis process had five steps:</u></i>
1. <i><u>I removed duplicated rows. The dataset has no empty values and one exact duplicate, leaving 5,002 pairs.</u></i>
2. <i><u>I split the data into a training part and a test part by unique hate speech, so that no hate speech appears in both parts.</u></i>
3. <i><u>I counted the pairs for each target group.</u></i>
4. <i><u>I measured the length of every text in words and in characters.</u></i>
5. <i><u>I plotted the target counts as a bar chart and compared the length distributions with overlapping histograms.</u></i>

<i><u>What I found:</u></i>
- <i><u>**The target groups are clearly imbalanced.** Muslims account for about 27 percent of the pairs, while disabled people account for only about 4 percent.</u></i>
- <i><u>**Counter-narratives are roughly twice as long** as the hate speech they answer: about 25 words compared with about 13 words on average, with a much wider spread of lengths.</u></i>
- <i><u>**There is no single correct answer.** 535 hate speeches have more than one counter-narrative, and one of them has 55.</u></i>

<i><u>That last observation points to the main challenge for both generation and evaluation. A model has to produce a reply that is specific, factual and respectful. At the same time, comparing it with only one human reply can underestimate a good answer that is simply worded differently.</u></i>

### <i><u>Version C: full worked example with the real results and their interpretation</u></i>

<i><u>**Results from the run:**</u></i>

| <i><u>Target</u></i> | <i><u>Pairs</u></i> | <i><u>Share</u></i> |
|---|---:|---:|
| <i><u>MUSLIMS</u></i> | <i><u>1,335</u></i> | <i><u>26.7%</u></i> |
| <i><u>MIGRANTS</u></i> | <i><u>956</u></i> | <i><u>19.1%</u></i> |
| <i><u>WOMEN</u></i> | <i><u>662</u></i> | <i><u>13.2%</u></i> |
| <i><u>LGBT+</u></i> | <i><u>617</u></i> | <i><u>12.3%</u></i> |
| <i><u>JEWS</u></i> | <i><u>594</u></i> | <i><u>11.9%</u></i> |
| <i><u>POC</u></i> | <i><u>352</u></i> | <i><u>7.0%</u></i> |
| <i><u>other</u></i> | <i><u>266</u></i> | <i><u>5.3%</u></i> |
| <i><u>DISABLED</u></i> | <i><u>220</u></i> | <i><u>4.4%</u></i> |

| <i><u>Length</u></i> | <i><u>Hate speech</u></i> | <i><u>Counter-narrative</u></i> |
|---|---:|---:|
| <i><u>Mean words</u></i> | <i><u>13.2</u></i> | <i><u>24.8</u></i> |
| <i><u>Median words</u></i> | <i><u>12</u></i> | <i><u>23</u></i> |
| <i><u>Standard deviation (words)</u></i> | <i><u>7.8</u></i> | <i><u>11.4</u></i> |
| <i><u>Longest text (words)</u></i> | <i><u>68</u></i> | <i><u>149</u></i> |
| <i><u>Mean characters</u></i> | <i><u>75.5</u></i> | <i><u>140.7</u></i> |
| <i><u>Median characters</u></i> | <i><u>66</u></i> | <i><u>130</u></i> |

<i><u>Other results:</u></i>
- <i><u>**Cleaning:** 5,003 rows loaded, 1 exact duplicate removed, leaving 5,002 pairs.</u></i>
- <i><u>**Unique hate speeches:** 3,718. Of these, 535 have more than one counter-narrative; the maximum is 55.</u></i>
- <i><u>**Split:** 4,052 training rows (2,974 unique hate speeches) and 950 test rows (744 unique hate speeches), with **0 overlap**. The test subset has 100 hate speeches.</u></i>

<i><u>**Interpretation:**</u></i>
- <i><u>**Imbalance (Figure 1).**</u></i>
  - <i><u>The largest group (Muslims, 1,335) is about six times the smallest (disabled people, 220).</u></i>
  - <i><u>In the 100-item test subset only a handful of items belong to the small groups.</u></i>
  - <i><u>Small groups also have fewer training examples for retrieval in Section 3.</u></i>
- <i><u>**Length (Figure 2).**</u></i>
  - <i><u>Replies are 1.9 times as long as the attacks, and their lengths vary more (standard deviation 11.4 versus 7.8 words; the longest reply has 149 words).</u></i>
  - <i><u>Three quarters of the expert replies have 30 words or fewer, so the prompt's limit of "2 to 3 sentences, maximum 60 words" fits the human style.</u></i>
- <i><u>**One-to-many.** 535 hate speeches have several valid replies. We therefore evaluate against **all** references for each hate speech, and we expect word-overlap scores such as BLEU to be low even for good replies.</u></i>

<i><u>**Filled-in short answer:** Version A above already uses these real numbers.</u></i>

---
# Section 2: Instruction-Based LLM Counter-Narrative Generation (40 marks)

## 2a: Build an instruction-based counter-narrative generator (20 marks)
*Brief: initialise an open-source instruction LLM and write a prompt template to generate non-aggressive, fact-bound counter-narratives on a test subset.*

### Choice of LLM

**What the brief requires:** open-source, **instruction-tuned**, inference only (no training).

**Why instruction-tuned and decoder-only (course arguments):**
- **Instruction tuning:** instruction-tuned models were fine-tuned on instruction–response pairs, so they "follow natural-language commands more reliably" (L6 p.43).
- **Decoder-only:**
  - Decoder-only models (GPT, LLaMA, …) generate text autoregressively, one token after another: P(X) = Π P(xᵢ | x₁…xᵢ₋₁) (L3 p.91; L5 p.12).
  - Encoder-only models like BERT are not suitable: "Generation is difficult", "Prompting is not possible" (L6 p.19).
- **Pretrained model:** training our own LLM is "not feasible" (L6 p.23), so we use a pretrained one (the "pre-train, prompt, predict" paradigm, L6 p.41).

**Candidates considered:**

| Model | Size | Where it appears | Verdict |
|---|---|---|---|
| `bigscience/bloom-560m` | 0.56B | WS6b, WS7b | ✗ A **base** model, not instruction-tuned |
| `bigscience/bloomz-560m` | 0.56B | Lab7 (default) | ✗ Too weak: in Lab7 it gave the same answer for every input, whatever the prompting method |
| `google/flan-t5-large` | 0.8B | Lab7 Part A | ✗ Encoder–decoder trained mostly on short-answer tasks; usually gives very short outputs |
| `TinyLlama/TinyLlama-1.1B-Chat-v1.0` | 1.1B | Lab7 option | Possible, but smaller, so expected to follow detailed rules less well |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B | (not in the course) | Fits easily, but smaller, so expected to follow detailed rules less well |
| **`Qwen/Qwen2.5-3B-Instruct`** | **3.1B** | (not in the course) | ✓ **Chosen for the T4** |
| `Qwen/Qwen2.5-7B-Instruct` | 7.6B | (not in the course) | Stronger, but on a T4 it leaves no room for a larger judge (it is used as the judge instead) |

**Why Qwen2.5-3B-Instruct:**
- It is open-source, with no sign-up or access request. The licence is the Qwen Research licence (rather than Apache 2.0), which allows non-commercial use such as coursework.
- It is instruction-tuned with a chat format. Being newer and larger than the course models, it is expected to follow detailed rules (persona, tone, length, "only accepted facts") better. The course models were not tested in this notebook; in Lab7, `bloomz-560m` gave the same answer to every input.
- In 4-bit it needs only about 2 GB of GPU memory. That leaves room on the T4 for a **larger** judge model (7B), which the evaluation needs (see 2b).
- The tutor said "a small (low-parameter) model is fine", and a 3B model is small and fast on a T4.

**4-bit quantisation (the Lab7 configuration).**
- Normally every model weight is stored as a 16-bit number. **4-bit quantisation** stores each weight in 4 bits instead, about **4 times less memory**, with only a small loss of quality.
- **NF4** ("NormalFloat 4") is a 4-bit format designed for how model weights are distributed.
- **Double quantisation** also compresses the scaling constants, which saves a little more memory.
- Calculations are still done in 16 bits (`float16` on the T4).
- These are exactly the settings used in Lab7 (`BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", ...)`). WS6b used the 8-bit version of the same idea.

**Why the Hugging Face pipeline and not Ollama:** the tutor allowed Ollama, but it needs a separate server process, which is awkward in Colab. The `transformers` pipeline is exactly what Lab7 uses.

In [ ]:
# 4-bit loading settings, the same as in Lab7
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # store weights in 4 bits (about 4x less memory)
    bnb_4bit_use_double_quant=True,         # also compress the scaling constants
    bnb_4bit_quant_type="nf4",              # "NormalFloat 4", a 4-bit format made for model weights
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,   # calculations in 16 bits (float16 on the T4)
)

def load_llm(model_name):
    """Load a tokenizer + 4-bit model and wrap them in a text-generation pipeline (as in Lab7)."""
    print("Loading", model_name, "in 4-bit ...")
    llm_tokenizer = AutoTokenizer.from_pretrained(model_name)
    llm_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",                  # put the model on the GPU automatically
    )
    llm_pipeline = pipeline("text-generation", model=llm_model, tokenizer=llm_tokenizer)
    return llm_tokenizer, llm_pipeline

tokenizer, generator = load_llm(MODEL_NAME)

### Prompt design

The prompt follows the **starter kit (SK)** structure (Task → Rules → Comment → "Counter-narrative:"), improved with the prompting guidelines from L6. Each part has a reason:

| Prompt part | What it says | Course source |
|---|---|---|
| **System message (persona)** | "You are an experienced counter-speech operator at an anti-discrimination NGO…" | AUTOMAT **A**ct as… (L6 p.66); a role influences style (L6 p.58) |
| **Task with a clear verb** | "Write a counter-narrative response to…" | Instruction with a verb (L6 p.48); AUTOMAT **T**argeted action |
| **Target group** | "The comment targets Muslims." | Context element (L6 p.46): helps the model address the right group |
| **Rules** | Respond to the specific claim; calm and non-aggressive; only widely accepted facts | The brief's "non-aggressive, fact-bound"; AUTOMAT **M**ode/tone; SK rules |
| **Fail case** | "If you are not sure a fact is correct, do not give numbers; argue from shared values and empathy instead" | Give fail cases (L6 p.51); AUTOMAT **A**typical cases; reduces hallucination (L7 p.43) |
| **Positive wording** | "Use neutral, respectful words" instead of only "don't…" | LLMs tend to do what they are told *not* to do (L6 p.51) |
| **Length limit** | "2 to 3 sentences (maximum 60 words)" | Be precise ("2–3 sentences", not "briefly", L6 p.50); matches the human length (~25 words) |
| **Output indicator** | "Provide only the counter-narrative…" + ending with "Counter-narrative:" | Output format / output indicator (L6 pp.46, 58); WS6b step 2 "specify the output" |

**Zero-shot:** no examples are given. This is our Section 2 baseline (L6 p.53; L3 p.93). The SK also calls itself "a zero-shot baseline".

**Shared rules:** the rules are stored in one variable `RULES` that Section 3 reuses unchanged, so the *only* difference between the two methods is the added examples.

In [ ]:
# Nicer wording for the target labels used in the dataset
TARGET_DESCRIPTIONS = {
    "MUSLIMS": "Muslims",
    "MIGRANTS": "migrants",
    "WOMEN": "women",
    "LGBT+": "LGBT+ people",
    "JEWS": "Jewish people",
    "POC": "people of colour",
    "DISABLED": "disabled people",
    "other": "a group of people",
}

# System message = persona and general behaviour (AUTOMAT "Act as" and "Mode")
SYSTEM_PROMPT = (
    "You are an experienced counter-speech operator at an anti-discrimination NGO. "
    "You write calm, respectful replies to hateful online comments. "
    "Your replies correct false claims with widely accepted facts, "
    "appeal to empathy and shared values, and never insult anyone."
)

# Rules shared by Section 2 and Section 3 (so the only difference in Section 3 is the examples)
RULES = """Rules:
- Respond directly to the specific claim made in the comment.
- Be calm, respectful and non-aggressive. Use neutral, respectful words (no slurs or insults).
- Only use facts that are widely accepted.
  If you are not sure a fact is correct, do not give numbers; argue from shared values and empathy instead.
- Write 2 to 3 sentences (maximum 60 words).
- Provide only the counter-narrative, without any introduction or label."""

def build_zero_shot_prompt(hate_speech, target):
    """Build the zero-shot user prompt for one hate speech (Section 2)."""
    target_text = TARGET_DESCRIPTIONS.get(target, "a group of people")   # .get: default if unknown
    prompt = f"""Task: Write a counter-narrative response to the hateful comment below.
The comment targets {target_text}.

{RULES}

Comment:
{hate_speech}

Counter-narrative:"""
    return prompt

### Generation settings and helper functions

**Decoding: greedy (`do_sample=False`).**
- Greedy decoding always takes the most probable next token. It is the `torch.argmax` route in WS7b; sampling is the `torch.multinomial` route.
- **Reproducible:** the same input always gives the same output, so Section 2 and Section 3 are compared fairly.
- **Less hallucination:** "increasing diversity (temperature) leads to more hallucinations" (L7 p.44). The brief wants fact-bound replies.
- **Downside:** greedy output can be repetitive or bland (L7 p.47).
- The SK used `temperature=0.7` (sampling). We deliberately do not, for the reasons above.
- **Harmless warning:** the model's default settings include a temperature. With `do_sample=False` it is ignored, and transformers may warn about this (Lab7 shows the same warning). We hid these warnings in the imports cell.

**Helper functions:**
- **`generate_text(...)`** builds the chat messages, applies the chat template and runs the pipeline.
  - `apply_chat_template` adds the model's **hidden tags** (L6 p.43: "Chat LLMs do not do next word completion on input!"). For Qwen they look like `<|im_start|>system … <|im_end|>`, `<|im_start|>user … <|im_end|>` and `<|im_start|>assistant`, where the answer starts.
  - `return_full_text=False` returns only the *new* text, so we don't need Lab7's `.replace(prompt, "")` trick.
  - The same function is reused for the judge later.
- **`clean_output(...)`** removes leftovers such as a repeated "Counter-narrative:" label or quotation marks. This is the same idea as Lab7's output cleaning.
- **`is_refusal(...)`** detects refusals. The tutor said: if the LLM refuses because of its safety training, **record the refusal rate**.
  - It is a simple keyword check on the start of the reply ("I can't help", "As an AI", …).
  - We avoid plain "I'm sorry", because real counter-narratives often start with "I'm sorry you feel that way".
  - An empty output also counts as a failure.

In [ ]:
def generate_text(llm_tokenizer, llm_pipeline, system_prompt, user_prompt, max_new_tokens):
    """Send one prompt to an instruction-tuned LLM and return only the generated answer."""
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    # Add the model's hidden chat tags (L6 p.43)
    prompt_text = llm_tokenizer.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True)

    outputs = llm_pipeline(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=False,          # greedy decoding (see explanation above)
        return_full_text=False,   # only return the new text, not the prompt
    )
    answer = outputs[0]["generated_text"]
    return answer.strip()

def clean_output(text):
    """Remove labels/quotes the model sometimes adds around its answer."""
    text = text.strip()
    for label in ["Counter-narrative:", "Counter-Narrative:", "Counter narrative:", "Response:"]:
        if text.startswith(label):
            text = text[len(label):]          # cut the label off the front
            text = text.strip()
    text = text.strip('"“”')                  # remove straight or curly quotes at the start/end
    return text.strip()

REFUSAL_PHRASES = [
    "i can't help", "i cannot help", "i can't assist", "i cannot assist",
    "i won't", "i will not", "as an ai", "i'm unable to",
]

def is_refusal(text):
    """Return True if the reply looks like a refusal (or is empty)."""
    if text.strip() == "":
        return True
    start = text[:150].lower()             # only look at the beginning of the reply
    start = start.replace("’", "'")        # treat curly apostrophes like straight ones
    for phrase in REFUSAL_PHRASES:
        if phrase in start:
            return True
    return False

### The test subset

From the test hate speeches we draw `N_TEST` at random with the standard pandas `.sample(...)`. For each one we keep:
- the hate speech;
- its target;
- **all** of its human counter-narratives as references. Several references are fairer for BLEU, ROUGE and BERTScore, because there are many valid ways to reply (see 1c).

The same subset is used in Section 2 and Section 3, so the comparison is fair.

In [ ]:
test_subset = test_hate_speech.sample(n=N_TEST, random_state=RANDOM_SEED)

test_items = []
for hate_speech, target in zip(test_subset["HATE_SPEECH"], test_subset["TARGET"]):
    rows_for_this_hate_speech = test_df[test_df["HATE_SPEECH"] == hate_speech]
    item = {
        "hate_speech": hate_speech,
        "target": target,
        "references": list(rows_for_this_hate_speech["COUNTER_NARRATIVE"]),   # all human replies
    }
    test_items.append(item)

print("Number of test items used:", len(test_items))
print("Example item:", test_items[0])

### Sanity check on one example

Always try one example before running everything (WS6b does the same). We use a **training** example, because the brief keeps the test data for the final evaluation only. If the output is empty or looks like garbage, check that the runtime is set to a T4 GPU.

In [ ]:
# Try the zero-shot prompt on one TRAINING example (the test data is kept for the final evaluation)
example_row = train_df.iloc[0]                     # the first row of the training data
example_prompt = build_zero_shot_prompt(example_row["HATE_SPEECH"], example_row["TARGET"])
example_answer = generate_text(tokenizer, generator, SYSTEM_PROMPT, example_prompt, MAX_NEW_TOKENS)
example_output = clean_output(example_answer)

print("HATE SPEECH:     ", example_row["HATE_SPEECH"])
print("TARGET:          ", example_row["TARGET"])
print("GENERATED:       ", example_output)
print("HUMAN REFERENCE: ", example_row["COUNTER_NARRATIVE"])
print("Refusal?         ", is_refusal(example_output))

### Generate counter-narratives for the whole test subset (zero-shot)

A simple `for` loop generates for each item one by one (as in SK and Lab7). `tqdm` shows a progress bar with the elapsed time. The outputs are saved to a CSV file (an SK idea): if you need to evaluate again later, you can reload them instead of generating again.

In [ ]:
def make_results_table(items, predictions):
    """Put inputs, references and predictions in one DataFrame (one row per test item)."""
    rows = []
    for item, prediction in zip(items, predictions):
        rows.append({
            "hate_speech": item["hate_speech"],
            "target": item["target"],
            "human_reference": item["references"][0],
            "number_of_references": len(item["references"]),
            "prediction": prediction,
            "refusal": is_refusal(prediction),
        })
    return pd.DataFrame(rows)

baseline_predictions = []
for item in tqdm(test_items, desc="Zero-shot generation"):
    prompt = build_zero_shot_prompt(item["hate_speech"], item["target"])
    answer = generate_text(tokenizer, generator, SYSTEM_PROMPT, prompt, MAX_NEW_TOKENS)
    baseline_predictions.append(clean_output(answer))

baseline_df = make_results_table(test_items, baseline_predictions)
baseline_df.to_csv("section2_zero_shot_outputs.csv", index=False)
print("Saved section2_zero_shot_outputs.csv")

## 2b: Evaluate counter-narratives (10 marks)
*Brief: evaluate your generated responses against human ground truths using quantitative NLP metrics like BLEU, ROUGE and BERTScore. Also evaluate using rubric-based scores on the generated counter-narratives.*

The tutor described **two kinds of evaluation**: **reference-based** (compare with the human counter-narrative) and **rubric-based** (a scale, applied through a prompt). We do both, plus the refusal rate.

### Part 1: Reference-based metrics

| Metric | What it measures | Course link |
|---|---|---|
| **BLEU** | Word n-gram (1–4 word sequences) **precision** against the references, with a penalty for too-short outputs. Exact word overlap. | Lab7 uses `evaluate.load("bleu")`; precision (L4 p.31) |
| **ROUGE-1 / ROUGE-2** | Overlap of single words / word pairs; **recall-oriented** (`evaluate` reports the F-measure) | Recall / F1 (L4 pp.31–32) |
| **ROUGE-L** | The **longest common subsequence**: shared words in the same order | n/a |
| **BERTScore-F1** | Embeds both texts with a BERT-style model, matches each word to its most similar word in the other text by **cosine similarity**, and reports P/R/F1. Measures **meaning**, so paraphrases score well. | Cosine (L3 p.31); BERT contextual embeddings (L3 p.85); the tutor's "semantic scores" |

**Why all four:**
- BLEU and ROUGE only count identical words. L7 p.45 warns that such metrics "focus on text fluency, not factual accuracy".
- A good counter-narrative can use completely different words from the human one, so BERTScore adds a *meaning-based* view.
- Even so, all of them only compare with the reference. That is why we also need the rubric.

**Details:**
- **Lowercasing:** BLEU is case-sensitive, so we lowercase both sides, as Lab7 did. ROUGE lowercases by itself.
- **Multiple references:** each prediction is compared with *all* human counter-narratives for that hate speech.
- **`rescale_with_baseline=True`** for BERTScore: raw BERTScores are all squeezed between about 0.8 and 0.9. Rescaling spreads them out so differences are easier to see. 0 then means "as similar as two unrelated sentences", and values can be slightly negative.
- **Expected values:** BLEU will be **low** (often below 0.05). That is normal for open-ended generation, because there are many valid replies (1c).

In [ ]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def lowercase_all(texts):
    """Return a new list with every text in lowercase."""
    lowered = []
    for text in texts:
        lowered.append(text.lower())
    return lowered

def compute_reference_scores(predictions, items):
    """Compute BLEU, ROUGE and BERTScore against all human references.
    Returns (dictionary of average scores, list of per-example BERTScore F1)."""
    references = []           # one list of human replies per test item
    lower_references = []     # the same, in lowercase (for BLEU)
    for item in items:
        references.append(item["references"])
        lower_references.append(lowercase_all(item["references"]))

    bleu_result = bleu_metric.compute(predictions=lowercase_all(predictions),
                                      references=lower_references)
    rouge_result = rouge_metric.compute(predictions=predictions, references=references)
    bert_result = bertscore_metric.compute(predictions=predictions, references=references,
                                           lang="en", rescale_with_baseline=True)

    per_example_bertscore = bert_result["f1"]      # one F1 value per test item
    scores = {
        "BLEU": bleu_result["bleu"],
        "ROUGE-1": rouge_result["rouge1"],
        "ROUGE-2": rouge_result["rouge2"],
        "ROUGE-L": rouge_result["rougeL"],
        "BERTScore-F1": sum(per_example_bertscore) / len(per_example_bertscore),
    }
    return scores, per_example_bertscore

### Part 2: Rubric-based scores (LLM as a judge, strict 1–5 version)

**The rubric.** The tutor and classmates reported that you may **define your own rubric**, as long as you explain what each score means; classmates reported that a 1–3 scale was accepted. In a first run with a 1–3 scale, however, the judge gave **every** zero-shot reply full marks, so it could not separate good from weak replies. The judge therefore uses a **1–5 scale with sharp anchors**:

| Score | relevance | tone | factuality | clarity |
|---|---|---|---|---|
| **5** | Directly refutes the specific claim with a concrete argument or example | Calm, respectful and empathetic; not preachy | Every statement accurate, or clearly presented as opinion or values | Clear, concise and well structured |
| **4** | Addresses the specific claim, but with a general argument | Respectful but somewhat lecturing or preachy | Accurate overall, but includes a broad generalisation | Clear, with minor awkwardness |
| **3** | Partly addresses the claim; mostly a general defence of the group | Noticeably moralising or condescending | Vague claims presented as facts that cannot be checked | Understandable but wordy or awkward |
| **2** | Generic statement about respect or tolerance that ignores the claim | Harsh or sarcastic | At least one doubtful claim or statistic | Hard to follow |
| **1** | Off-topic, or agrees with or repeats the hateful claim | Insulting or aggressive | Clearly false or invented facts or statistics | Incoherent or ungrammatical |
| **Source** | Peers/tutor: "does it actually counter the hate speech" | Brief: "non-aggressive"; style (L7 p.41) | Brief: "fact-bound"; hallucination (L7 p.43) | Fluency (L7 p.41) |

**Who scores?** The tutor allowed a prompt-based scale (an LLM judge), human scoring, or both. "LLM can be asked to judge many things!" (L7 p.27). We use **only the LLM judge**, so that all 200 replies (100 × 2 methods) are scored the same way and nothing has to be scored by hand.

**Choices for the judge:**
- **Qwen2.5-7B-Instruct, more than twice the size of the generator (3B).**
  - Judging is harder than writing a short reply, so the judge should be the stronger model.
  - A model that judges its own outputs tends to favour them (self-preference bias). Using a separate, larger model reduces this. It is still the same model family, so some bias may remain.
  - **This bias affects Section 2 and Section 3 equally,** because both use the same writer. The *comparison* stays fair.
- **Strictness against the "ceiling effect".** Language-model judges tend to be lenient: they like fluent, polite text and give it top marks. Three measures counter this:
  1. **Sharper anchors.** Every score has a concrete meaning. A polite but generic "everyone deserves respect" reply now gets relevance 2, not the top score.
  2. **A calibration instruction.** The prompt says that most replies are acceptable but not excellent: 5 is rare, 3 is acceptable.
  3. **Critique first.** The judge first writes one sentence naming the main weakness of the reply, and only then gives the scores. Reasoning before answering is chain-of-thought prompting (L6 p.55), and asking for a weakness makes the judge look critically. The critiques are saved, so you can check *why* a reply got its scores.
- **One reply at a time ("pointwise")**: the judge never sees the two methods side by side, so the order of the answers cannot bias it.
- **Reference-free**: the judge does not see the human reply. The rubric measures quality *by itself*; similarity to the reference is already covered by Part 1.
- **JSON output**, as recommended "for downstream processing" (L6 p.58): `{"critique": ..., "relevance": ..., "tone": ..., "factuality": ..., "clarity": ...}`.
  - The critique comes first, so the model writes its reasoning before the numbers.
  - The answer is read with the standard `json.loads`. Answers that cannot be read are counted and left out of the averages.
- **Greedy decoding** makes the scores reproducible.
- **A score table** (after the Section 2 results) shows how often each score from 1 to 5 was given. If the judge still gave almost only 5s, the rubric would again be at the ceiling.
- **Limitation: the judge was adjusted after looking at test outputs.** The strict judge was designed after a first judge (1–3 scale) gave all 100 test replies full marks. The replies did not change and both methods are judged identically, so the comparison stays fair, but the judge was adjusted while looking at test outputs. Better practice: set aside a small development set from the training data (for example 20–30 hate speeches), design the prompts and the judge on it, and run the test set only once.

**One function for all measures.** `evaluate_method(...)` runs the reference metrics and the judge for one method, adds the per-example scores and critiques to its results table, saves it as CSV and returns all averages in one dictionary. Section 3 calls the same function, so both methods are measured identically.

In [ ]:
judge_tokenizer, judge_generator = load_llm(JUDGE_MODEL_NAME)

# The strict rubric: criterion -> what 5, 4, 3, 2 and 1 mean
RUBRIC = {
    "relevance": "5 = directly refutes the specific claim with a concrete argument or example, "
                 "4 = addresses the specific claim but with a general argument, "
                 "3 = partly addresses the claim, mostly a general defence of the group, "
                 "2 = generic statement about respect or tolerance that ignores the claim, "
                 "1 = off-topic, or agrees with or repeats the hateful claim",
    "tone": "5 = calm, respectful and empathetic, not preachy, "
            "4 = respectful but somewhat lecturing or preachy, "
            "3 = noticeably moralising or condescending, "
            "2 = harsh or sarcastic, "
            "1 = insulting or aggressive",
    "factuality": "5 = every statement is accurate, or clearly presented as opinion or values, "
                  "4 = accurate overall but includes a broad generalisation, "
                  "3 = vague claims presented as facts that cannot be checked, "
                  "2 = at least one doubtful claim or statistic, "
                  "1 = clearly false or invented facts or statistics",
    "clarity": "5 = clear, concise and well structured, "
               "4 = clear with minor awkwardness, "
               "3 = understandable but wordy or awkward, "
               "2 = hard to follow, "
               "1 = incoherent or ungrammatical",
}

VALID_SCORES = [1, 2, 3, 4, 5, "1", "2", "3", "4", "5"]   # numbers, or numbers written as text

# The exact answer format we ask for (a normal string, so the curly brackets need no escaping).
# "critique" comes first, so the judge writes its reasoning before the scores.
JUDGE_JSON_FORMAT = ('{"critique": "<one sentence>", "relevance": <1-5>, "tone": <1-5>, '
                     '"factuality": <1-5>, "clarity": <1-5>}')

def build_judge_prompt(hate_speech, counter_narrative):
    """Build the prompt that asks the judge to critique and score one counter-narrative."""
    rubric_lines = ""
    for criterion, description in RUBRIC.items():
        rubric_lines += f"- {criterion}: {description}\n"

    prompt = f"""You are a strict expert evaluator of counter-narratives (replies to hate speech).
Most replies are acceptable but not excellent, so use the whole scale:
5 = excellent (rare), 4 = good, 3 = acceptable, 2 = weak, 1 = very poor.

Hateful comment:
{hate_speech}

Counter-narrative to evaluate:
{counter_narrative}

Scoring guide:
{rubric_lines}
First write one sentence of critique that names the main weakness of the counter-narrative.
Then give the scores. Answer with JSON only, in exactly this format:
{JUDGE_JSON_FORMAT}"""
    return prompt

def parse_judge_answer(judge_answer):
    """Read the judge's JSON answer. Returns (scores, critique).
    Missing or invalid scores become None."""
    scores = {}
    for criterion in RUBRIC:
        scores[criterion] = None                  # default: no valid score found
    critique = ""

    start = judge_answer.find("{")                # position of the first "{"
    end = judge_answer.rfind("}")                 # position of the last "}"
    if start == -1 or end == -1:
        return scores, critique                   # no JSON in the answer

    json_text = judge_answer[start:end + 1]       # cut out the part between { and }
    try:
        data = json.loads(json_text)
    except json.JSONDecodeError:
        return scores, critique                   # the JSON was broken
    if not isinstance(data, dict):
        return scores, critique

    critique = str(data.get("critique", ""))
    for criterion in RUBRIC:
        value = data.get(criterion)
        if value in VALID_SCORES:
            scores[criterion] = int(value)
    return scores, critique

def judge_all(items, predictions):
    """Ask the judge to critique and score every prediction.
    Returns (DataFrame of scores with rows = items and columns = criteria, list of critiques)."""
    all_scores = []
    all_critiques = []
    n_failed = 0
    for i in tqdm(range(len(items)), desc="Judging"):
        prompt = build_judge_prompt(items[i]["hate_speech"], predictions[i])
        answer = generate_text(judge_tokenizer, judge_generator, None, prompt, JUDGE_MAX_NEW_TOKENS)
        scores, critique = parse_judge_answer(answer)
        if None in scores.values():
            n_failed += 1
        all_scores.append(scores)
        all_critiques.append(critique)
    print("Judge answers that could not be read completely:", n_failed)
    judgements = pd.DataFrame(all_scores, dtype=float)    # None becomes NaN ("missing")
    return judgements, all_critiques

def evaluate_method(predictions, results_df, csv_name):
    """Score one method with every measure (Sections 2 and 3 both use this function).
    Returns (dictionary of average scores, per-example BERTScore F1, judge scores, critiques)."""
    reference_scores, bertscores = compute_reference_scores(predictions, test_items)
    judgements, critiques = judge_all(test_items, predictions)

    # Save the per-example rubric scores and critiques next to the replies
    for criterion in RUBRIC:
        results_df["rubric_" + criterion] = judgements[criterion]
    results_df["rubric_critique"] = critiques
    results_df.to_csv(csv_name, index=False)

    # All averages in one dictionary: reference scores, refusal rate, rubric (1-5)
    scores = {}
    scores.update(reference_scores)                        # BLEU, ROUGE, BERTScore
    scores["Refusal rate"] = results_df["refusal"].mean()  # mean of True/False = share of refusals
    criterion_averages = judgements.mean()                 # one average per criterion (gaps skipped)
    for criterion in RUBRIC:
        scores["Rubric " + criterion] = criterion_averages[criterion]
    scores["Rubric overall (1-5)"] = criterion_averages.mean()
    return scores, bertscores, judgements, critiques

In [ ]:
# Score the zero-shot replies with every measure
section2_scores, baseline_bertscores, baseline_judgements, baseline_critiques = evaluate_method(
    baseline_predictions, baseline_df, "section2_zero_shot_outputs.csv")

section2_table = pd.DataFrame({"Section 2: zero-shot": section2_scores})
display(section2_table.round(3))

# How often was each score given? (the judge should not give everyone top marks)
score_table = pd.DataFrame({
    "relevance": baseline_judgements["relevance"].value_counts(),
    "tone": baseline_judgements["tone"].value_counts(),
    "factuality": baseline_judgements["factuality"].value_counts(),
    "clarity": baseline_judgements["clarity"].value_counts(),
})
score_table = score_table.fillna(0).astype(int)    # a score that never occurs gets 0
score_table = score_table.sort_index()             # scores in order 1, 2, 3, 4, 5
score_table.index = score_table.index.astype(int)  # show 3 instead of 3.0
score_table.index.name = "score"
print("How often each score (1-5) was given in Section 2:")
display(score_table)

## 2c: Explanation (10 marks)

**Question:** Briefly explain your instruction-based LLM generation method and the results. Include: why you chose the LLM; how you designed the prompt; which evaluation scores you used and why; one strength and one limitation of the generated counter-narratives. (Max **50 words**, in your own words.)

### Version A: short answer (44 words)

```
I used Qwen2.5-3B-Instruct (open-source, instruction-tuned, fits the T4). The prompt sets a persona, rules, fail case and length limit. BLEU/ROUGE check word overlap (0.008/0.11), BERTScore meaning (0.25), a strict 1–5 rubric quality. Strength: respectful replies (tone 4.0/5), no refusals. Limitation: generic arguments (relevance 3.1/5).
```

<i><u>**Study aid – not part of the core workbook.** Skip this part when you write your own submission.</u></i>

### <i><u>Version B: long explanation (study notes)</u></i>

<i><u>For the generation method I used Qwen2.5-3B-Instruct, an open-source large language model that has been instruction-tuned. This means it was trained further on pairs of instructions and responses, so it follows written instructions reliably. It is a decoder-only model: it writes text one word piece at a time, which is exactly what a generation task needs. An encoder-only model such as BERT cannot generate fluent text. The smallest model from the course lab was not strong enough: it gave the same answer to every input. I therefore chose a somewhat larger model with three billion parameters and loaded it in 4-bit precision, the same technique as in the course lab, so that it fits on the T4 graphics card together with a larger judge model.</u></i>

<i><u>The prompt follows the structure of the starter kit:</u></i>
- <i><u>a task;</u></i>
- <i><u>a list of rules;</u></i>
- <i><u>the hateful comment;</u></i>
- <i><u>the label "Counter-narrative:" where the answer should begin.</u></i>

<i><u>A system message gives the model the role of an experienced counter-speech operator at an anti-discrimination organisation. The rules ask the model to:</u></i>
- <i><u>respond to the specific claim;</u></i>
- <i><u>stay calm and respectful;</u></i>
- <i><u>use only widely accepted facts, and argue from shared values whenever it is not sure about a fact;</u></i>
- <i><u>write two or three sentences;</u></i>
- <i><u>output only the reply.</u></i>

<i><u>The rules describe what the model should do rather than what it should avoid, because language models tend to do the very thing they are told not to do. I used greedy decoding, which always picks the most likely next word, so the results are reproducible and less likely to contain invented facts.</u></i>

<i><u>I evaluated the replies in three ways:</u></i>
- <i><u>**Word overlap.** BLEU and ROUGE count the words and word sequences that a reply shares with the human replies. They are standard measures, but strict: a good reply can use completely different words.</u></i>
- <i><u>**Meaning.** BERTScore compares the meaning of the words using contextual embeddings, so a reply that says the same thing in other words is still rewarded.</u></i>
- <i><u>**A strict rubric scored by a larger judge model.** None of the measures above checks whether a reply is respectful or factual. A larger language model (Qwen2.5-7B-Instruct) therefore scored every reply from one to five on relevance, tone, factuality and clarity, after first writing one sentence about the reply's main weakness.</u></i>

<i><u>I also recorded how often the model refused to answer.</u></i>

<i><u>**Why the rubric is strict.** In a first run, a one-to-three scale gave every reply full marks, so it could not tell good replies from weaker ones. I therefore made the scale finer, gave each score a precise meaning, told the judge that the top score should be rare, and asked it to criticise before scoring. This is the iterative prompt design taught in the course, applied to the judge. It worked for relevance, where the scores now spread from two to four.</u></i>

<i><u>**Results (100 test comments).**</u></i>
- <i><u>**Word overlap was very low:** BLEU 0.008, ROUGE-1 0.147, ROUGE-2 0.014 and ROUGE-L 0.108. The replies almost never share four-word sequences with the expert replies.</u></i>
- <i><u>**BERTScore was 0.254 after rescaling.** The replies are clearly closer in meaning to the expert replies than unrelated sentences would be, but they are far from paraphrases of them.</u></i>
- <i><u>**The model never refused** to answer.</u></i>
- <i><u>**On the strict rubric:**</u></i>
  - <i><u>Relevance averaged 3.09 out of 5. 87 of the 100 replies received 3, which means "partly addresses the claim, mostly a general defence of the group"; only 11 received 4.</u></i>
  - <i><u>Tone averaged 4.04 and clarity 4.05: almost every reply was respectful and clear, but slightly lecturing.</u></i>
  - <i><u>Factuality averaged 4.95, because the replies make almost no factual claims; they argue with values instead.</u></i>

<i><u>**Strength.** The replies were consistently calm, respectful and fluent, although every input was hateful, and the model never refused.</u></i>

<i><u>**Limitation.** Most replies were generic: they defended the group in general instead of rebutting the specific claim. The judge's critiques say this again and again, for example "The counter-narrative does not directly address the claim that immigrants are criminals."</u></i>

### <i><u>Version C: full worked example with the real results and their interpretation</u></i>

<i><u>**Results from this run (Section 2, zero-shot, 100 test items):**</u></i>

| <i><u>Metric</u></i> | <i><u>Score</u></i> |
|---|---:|
| <i><u>BLEU</u></i> | <i><u>0.008</u></i> |
| <i><u>ROUGE-1</u></i> | <i><u>0.147</u></i> |
| <i><u>ROUGE-2</u></i> | <i><u>0.014</u></i> |
| <i><u>ROUGE-L</u></i> | <i><u>0.108</u></i> |
| <i><u>BERTScore-F1 (rescaled)</u></i> | <i><u>0.254</u></i> |
| <i><u>Refusal rate</u></i> | <i><u>0.000</u></i> |
| <i><u>Rubric relevance (1–5)</u></i> | <i><u>3.09</u></i> |
| <i><u>Rubric tone (1–5)</u></i> | <i><u>4.04</u></i> |
| <i><u>Rubric factuality (1–5)</u></i> | <i><u>4.95</u></i> |
| <i><u>Rubric clarity (1–5)</u></i> | <i><u>4.05</u></i> |
| <i><u>Rubric overall (1–5)</u></i> | <i><u>4.03</u></i> |

<i><u>**How often each score was given (Section 2):**</u></i>

| <i><u>Score</u></i> | <i><u>relevance</u></i> | <i><u>tone</u></i> | <i><u>factuality</u></i> | <i><u>clarity</u></i> |
|---:|---:|---:|---:|---:|
| <i><u>2</u></i> | <i><u>2</u></i> | <i><u>0</u></i> | <i><u>0</u></i> | <i><u>0</u></i> |
| <i><u>3</u></i> | <i><u>87</u></i> | <i><u>0</u></i> | <i><u>0</u></i> | <i><u>0</u></i> |
| <i><u>4</u></i> | <i><u>11</u></i> | <i><u>96</u></i> | <i><u>5</u></i> | <i><u>95</u></i> |
| <i><u>5</u></i> | <i><u>0</u></i> | <i><u>4</u></i> | <i><u>95</u></i> | <i><u>5</u></i> |

<i><u>Judge answers that could not be read: 0 of 100.</u></i>

<i><u>**Real example (test item 1):**</u></i>
- <i><u>*Hate speech:* "Most immigrants in the U.S. are criminals."</u></i>
- <i><u>*Human reference:* "Not all immigrants are criminals. And it is important to consider that 13.5% of the US population is composed of migrants that are here legally."</u></i>
- <i><u>*Generated (zero-shot):* "Many immigrants contribute positively to society through hard work and innovation. We should focus on their contributions rather than generalizing based on a small minority."</u></i>
- <i><u>*First judge (1–3 scale):* 3 on every criterion.</u></i>
- <i><u>*Strict judge (1–5 scale):* relevance 3, tone 4, factuality 5, clarity 4.</u></i>
- <i><u>*Critique:* "The counter-narrative does not directly address the claim that immigrants are criminals."</u></i>

<i><u>**Interpretation:**</u></i>
- <i><u>**BLEU of 0.008 means almost no shared four-word sequences.** The model words its replies completely differently from the experts. This is expected for open-ended generation, and it is why BLEU alone cannot judge counter-narratives (L7 p.45).</u></i>
- <i><u>**ROUGE-1 of 0.147** means about one word in seven is shared. ROUGE-2 (0.014) shows that two-word sequences almost never match.</u></i>
- <i><u>**BERTScore of 0.254 (rescaled)** is clearly above 0, so the replies are on the topic of the references, but far from paraphrases of them.</u></i>
- <i><u>**A refusal rate of 0%:** safety training did not block this task, even with very offensive inputs.</u></i>
- <i><u>**The strict rubric now separates replies on relevance.** With the first (1–3) judge, every reply got full marks. Now 87 of 100 replies get relevance 3, 11 get 4 and 2 get 2. The example shows why: the reply praises immigrants' contributions instead of answering "criminals". The critiques name exactly this weakness for almost every reply. This is the "poorer performance" of zero-shot prompting on a complex task (L6 p.53), now measured.</u></i>
- <i><u>**Tone (4.04) and clarity (4.05) are high but not perfect.** Almost all replies get 4 ("respectful but somewhat lecturing"), which matches their "we should…" style.</u></i>
- <i><u>**Factuality is still near the top (4.95).** This is mostly genuine rather than judge leniency: the replies contain almost no factual claims, and the anchor gives 5 to statements "clearly presented as opinion or values". The fail-case rule in the prompt ("do not give numbers when not sure") worked.</u></i>
- <i><u>**Limitation of the evaluation:** tone, factuality and clarity hardly vary, so only relevance carries much information. The judge was not checked against human scores.</u></i>

<i><u>**Filled-in short answer:** Version A above already uses these real numbers.</u></i>

---
# Section 3: Improving the Counter-Narrative Generation Method (30 marks)

## 3a: Improve the method (15 marks)

### Choice of improvement: retrieval-augmented few-shot prompting

**Idea:**
- For each test hate speech, **find the 3 most similar hate speeches in the training data**, using sentence embeddings and cosine similarity.
- Put those hate speeches *with their expert-written counter-narratives* into the prompt as examples.
- Then ask for a new counter-narrative.

**The LLM judge is *not* the improvement.** It is part of the evaluation and scores both methods in exactly the same way.

The improvement combines two course techniques:
- **Few-shot prompting** (L6 p.54; L3 p.93): "a few (1–5) examples are added into the prompt; the model can generalise based on the examples".
- **Retrieve-Read RAG** (L7 p.5):
  1. query the knowledge base;
  2. get the top-k results;
  3. prepend them to the question;
  4. generate.
  - Our knowledge base is the training set, as the brief suggests ("training data for … retrieval examples").

**Why this improvement (the motivation):**
1. **Zero-shot weaknesses:** "poorer performance; hard to apply to complex tasks" (L6 p.53). Few-shot examples improve performance and show the expected format (L6 p.54).
2. **Choosing the examples well:** few-shot is "sensitive to the number and choice of examples" (L6 p.54). Retrieval gives *relevant* examples for each input instead of fixed random ones.
   - A hate speech about migrants and benefits gets expert replies to similar claims, with the kinds of arguments the experts used.
3. **Grounding against hallucination:** RAG makes the LLM "less likely to generate incorrect information", and "small models perform similarly to larger LLMs on factual tasks" (L7 p.6). The examples contain real expert arguments, which supports the brief's "fact-bound" requirement.
4. **No training needed:** RAG "uses models off-the-shelf" (L7 p.29), so it fits the inference-only rule.
5. **Supported by guidance:** the brief names semantic retrieval / RAG and few-shot examples, and the tutor listed "adding examples" and "RAG".
   - Classmates reported that RAG is *not required*, and that "a library is fine" if you use it.

**Alternatives considered:**

| Alternative | Why not chosen |
|---|---|
| Changing temperature (allowed by the tutor) | Weakest motivation: higher temperature gives more variety but "more hallucinations" (L7 p.44); it adds no knowledge |
| Chain-of-thought (L6 p.55) | Long outputs and more computation; designed for reasoning problems, not short persuasive replies |
| Fixed random few-shot examples (as in Lab7) | Simpler, but the examples may be about a different target and claim (the choice-sensitivity problem) |
| Web-search RAG (tutor option) | Needs a search API; web text is less reliable than expert-written replies |
| Fine-tuning / soft prompting (L6 p.61) | Not allowed: the assignment is inference only |

**What stays exactly the same (a fair comparison):**
- the model;
- the system prompt;
- the `RULES`;
- greedy decoding;
- `MAX_NEW_TOKENS`;
- the 100 test items;
- the metrics and the judge.

The **only** change is the retrieved examples in the prompt.

### Dense retrieval with Sentence Transformers

**How it works:**
1. **Embed** every training hate speech once (L7 p.14: "calculate the embedding of each document… can be done ahead of time").
   - The embedding model turns each text into a vector of 384 numbers that represents its meaning.
   - L7 p.13 describes a Sentence Transformer that collects the meaning in a special [DOC] token. `all-MiniLM-L6-v2` instead **averages all token vectors** ("mean pooling", L6 p.26). Both give one vector per text.
2. **Embed the query**, i.e. the test hate speech.
3. **Find the most similar training texts by cosine similarity:** q* = argmax_d cos(q, d) (L7 p.14).
   - cos(v, w) = (v · w) / (|v| |w|) (L3 p.31).
   - 1 = same direction (very similar meaning); 0 = unrelated.
   - The standard function `util.semantic_search` computes these similarities and returns the best matches, sorted from best to worst.
   - Checking every training vector is the simple search from L7 p.14. That is fine for about 4,000 training texts; approximate nearest-neighbour search (L7 p.15) is only needed for millions.

**Why dense rather than sparse retrieval:** sparse keyword retrieval (TF-IDF, L4 p.38; L7 p.11) needs the same words to match. Dense retrieval also finds similar meaning with different words (L7 pp.11–12), for example "foreigners" and "immigrants".

**Embedding model `all-MiniLM-L6-v2`:** a small (about 80 MB), fast and widely used general-purpose sentence-embedding model.

**Diversity filter:** in the training data, the same hate speech often appears with several counter-narratives. We skip a retrieved hate speech we have already used, so the 3 examples are about 3 *different* comments. This is a simple version of "filtering for diversity" (L7 p.22, MMR p.23).

In [ ]:
print("Loading embedding model", EMBEDDING_MODEL_NAME, "...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

train_texts = list(train_df["HATE_SPEECH"])           # the "documents" we search in
train_replies = list(train_df["COUNTER_NARRATIVE"])   # the expert reply for each document

# Embed all training hate speeches once (this is our "knowledge base").
# normalize_embeddings=True gives every vector length 1, which is standard for cosine similarity.
train_embeddings = embedder.encode(train_texts, convert_to_tensor=True,
                                   normalize_embeddings=True, show_progress_bar=True)
embedding_shape = tuple(train_embeddings.shape)
print("Embedding matrix shape (texts, numbers per text):", embedding_shape)

In [ ]:
def retrieve_examples(hate_speech, k=K_EXAMPLES):
    """Return the k most similar TRAINING pairs for a hate speech (dense retrieval, L7 p.14)."""
    query_embedding = embedder.encode(hate_speech, convert_to_tensor=True,
                                      normalize_embeddings=True)

    # semantic_search returns one result list per query; we have one query, so we take [0].
    # We ask for 5 candidates per example we need, because duplicates will be skipped.
    hits = util.semantic_search(query_embedding, train_embeddings, top_k=k * 5)[0]

    examples = []
    used_texts = set()
    for hit in hits:
        position = hit["corpus_id"]             # row number in the training data
        similar_text = train_texts[position]
        if similar_text in used_texts:          # diversity filter: skip a repeated hate speech
            continue
        used_texts.add(similar_text)
        examples.append({
            "hate_speech": similar_text,
            "counter_narrative": train_replies[position],
            "similarity": hit["score"],         # cosine similarity (L3 p.31)
        })
        if len(examples) == k:
            break
    return examples

### The retrieval-augmented prompt

- It uses the same Task line and the same `RULES` as Section 2. In between there is an "examples" block, introduced as "good counter-narratives written by NGO experts for similar comments" (a context element, L6 p.46).
- We tell the model to use the examples "as a guide for tone, length and kind of arguments, but write a new reply". Otherwise models sometimes just copy an example.
- **The most similar example goes *last*, next to the new comment.** LLMs pay most attention to the beginning and end of the input ("Lost in the Middle", L7 p.21), so the most useful example sits closest to the question.
- The format "Comment: … / Counter-narrative: …" mirrors Lab7's few-shot prompt ("Sentence: … / Category: …") and the SK prompt.

In [ ]:
def build_rag_prompt(hate_speech, target, examples):
    """Build the retrieval-augmented few-shot prompt (Section 3)."""
    target_text = TARGET_DESCRIPTIONS.get(target, "a group of people")

    ordered_examples = list(reversed(examples))     # most similar example last
    examples_text = ""
    for number, example in enumerate(ordered_examples, start=1):
        example_comment = example["hate_speech"]
        example_reply = example["counter_narrative"]
        examples_text += f"Example {number}\n"
        examples_text += f"Comment: {example_comment}\n"
        examples_text += f"Counter-narrative: {example_reply}\n\n"

    prompt = f"""Task: Write a counter-narrative response to the hateful comment below.
The comment targets {target_text}.

Here are examples of good counter-narratives written by NGO experts for similar comments.
Use them as a guide for tone, length and kind of arguments, but write a new reply that fits the new comment.

{examples_text}{RULES}

Comment:
{hate_speech}

Counter-narrative:"""
    return prompt

In [ ]:
# Same loop as Section 2; only the prompt differs (it now contains the retrieved examples)
rag_predictions = []
best_similarities = []        # similarity of the best retrieved example (for the analysis)

for item in tqdm(test_items, desc="Retrieval few-shot generation"):
    examples = retrieve_examples(item["hate_speech"])
    prompt = build_rag_prompt(item["hate_speech"], item["target"], examples)
    answer = generate_text(tokenizer, generator, SYSTEM_PROMPT, prompt, MAX_NEW_TOKENS)
    rag_predictions.append(clean_output(answer))
    best_similarities.append(examples[0]["similarity"])

rag_df = make_results_table(test_items, rag_predictions)
rag_df["best_example_similarity"] = best_similarities
rag_df.to_csv("section3_rag_outputs.csv", index=False)
print("Saved section3_rag_outputs.csv")

average_similarity = rag_df["best_example_similarity"].mean()
print(f"Average similarity of the best retrieved example: {average_similarity:.3f}")

## 3b: Evaluate the improved method (10 marks)
*Brief: evaluate the new outputs using the same metrics as Section 2b and display a side-by-side performance comparison table.*

We use **exactly the same function** as Section 2, `evaluate_method`, so both methods get identical measures. Lab7 also compared all prompting methods with the same metrics, one after another.

**What the comparison shows:**
- **Difference column (Section 3 − Section 2).** Higher is better for every row **except the refusal rate**, where lower is better.
- **Per-item counts:** for how many of the 100 items did BERTScore go up, and did the judge's relevance score go up or down? This shows whether a change in the average comes from many items or just a few.
- **A relevance table:** how many replies got each relevance score, zero-shot and retrieval side by side.
- **Side-by-side examples:** numbers can hide what changed, so we read some outputs.

In [ ]:
# Score the retrieval replies with exactly the same function as Section 2
section3_scores, rag_bertscores, rag_judgements, rag_critiques = evaluate_method(
    rag_predictions, rag_df, "section3_rag_outputs.csv")

In [ ]:
comparison = pd.DataFrame({
    "Section 2: zero-shot": section2_scores,
    "Section 3: retrieval few-shot": section3_scores,
})
comparison["Difference (3 - 2)"] = (comparison["Section 3: retrieval few-shot"]   # 3 minus 2
                                    - comparison["Section 2: zero-shot"])
print("Higher is better for all rows except 'Refusal rate' (lower is better).")
display(comparison.round(3))

In [ ]:
n_compared = len(test_items)                       # per-item comparison of BERTScore
n_improved = 0
for zero_shot_score, rag_score in zip(baseline_bertscores, rag_bertscores):
    if rag_score > zero_shot_score:
        n_improved += 1
print(f"Retrieval improved BERTScore for {n_improved} of {n_compared} test items.")

relevance_difference = rag_judgements["relevance"] - baseline_judgements["relevance"]   # per item
n_relevance_up = (relevance_difference > 0).sum()
n_relevance_down = (relevance_difference < 0).sum()
print(f"Strict-rubric relevance went up for {n_relevance_up} items and down for {n_relevance_down} items.")

relevance_table = pd.DataFrame({                   # how many replies got each relevance score
    "Section 2: zero-shot": baseline_judgements["relevance"].value_counts(),
    "Section 3: retrieval few-shot": rag_judgements["relevance"].value_counts(),
})
relevance_table = relevance_table.fillna(0).astype(int)    # a score that never occurs gets 0
relevance_table = relevance_table.sort_index()             # scores in order 1, 2, 3, 4, 5
relevance_table.index = relevance_table.index.astype(int)  # show 3 instead of 3.0
relevance_table.index.name = "relevance score"
display(relevance_table)

In [ ]:
for i in range(3):                                 # three test items, both methods side by side
    best_example = retrieve_examples(test_items[i]["hate_speech"])[0]   # the most similar training pair
    best_similarity = round(best_example["similarity"], 3)
    print("=" * 100)
    print("HATE SPEECH: ", test_items[i]["hate_speech"])
    print("HUMAN:       ", test_items[i]["references"][0])
    print("ZERO-SHOT:   ", baseline_predictions[i])
    print("   judge:     relevance", baseline_judgements["relevance"][i], "|", baseline_critiques[i])
    print("BEST EXAMPLE:", best_example["hate_speech"], "| similarity", best_similarity)
    print("   its reply: ", best_example["counter_narrative"])
    print("RETRIEVAL:   ", rag_predictions[i])
    print("   judge:     relevance", rag_judgements["relevance"][i], "|", rag_critiques[i])

## 3c: Explanation (5 marks)

**Question:** Briefly explain your improvement method and whether it improved the counter-narrative generation. Include: what improvement you applied; why you selected it; how the results changed compared with Section 2; one benefit and one limitation of the improved method. (Max **50 words**, in your own words.)

### Version A: short answer (48 words)

```
I added retrieval-augmented few-shot prompting: three similar training pairs (average similarity 0.80) guide each reply, because zero-shot replies were generic. BERTScore rose slightly (0.254 to 0.262; 57/100 items), but strict-rubric relevance fell (3.09 to 2.95). Benefit: expert-like, sometimes more direct replies. Limitation: replies copied short, generic expert style.
```

<i><u>**Study aid – not part of the core workbook.** Skip this part when you write your own submission.</u></i>

### <i><u>Version B: long explanation (study notes)</u></i>

<i><u>**What I changed.** I improved the method with retrieval-augmented few-shot prompting, which is a simple form of retrieval-augmented generation. For every test hate speech:</u></i>
1. <i><u>A sentence-embedding model turns the comment into a vector.</u></i>
2. <i><u>The method finds the three training comments with the highest cosine similarity, skipping a comment that was already chosen.</u></i>
3. <i><u>It places those three comments, with their expert-written counter-narratives, in the prompt as examples, with the most similar example last, closest to the new comment.</u></i>

<i><u>Everything else stayed exactly the same as in Section 2: the language model, the system message, the rules, greedy decoding, the length limit, the hundred test comments and all evaluation measures. Any difference in the results therefore comes from the examples.</u></i>

<i><u>**Why I chose it.** The zero-shot replies were polite but generic: the judge gave 87 of 100 of them a relevance of three, meaning they defended the group in general instead of answering the claim. Examples show the model what a specific reply looks like. Few-shot prompting is known to be sensitive to which examples are chosen, and retrieval solves this by picking examples about similar claims. Retrieval also grounds the reply in arguments that experts actually used, and it needs no training, which the assignment does not allow.</u></i>

<i><u>**How the results changed.**</u></i>
- <i><u>**Retrieval found close examples:** the most similar training comment had an average cosine similarity of 0.80.</u></i>
- <i><u>**Four of the five reference-based scores improved slightly,** so the replies became a little more similar to the expert replies:</u></i>
  - <i><u>BLEU went from 0.008 to 0.016;</u></i>
  - <i><u>ROUGE-2 from 0.014 to 0.016;</u></i>
  - <i><u>ROUGE-L from 0.108 to 0.110;</u></i>
  - <i><u>BERTScore from 0.254 to 0.262, improving for 57 of the 100 comments.</u></i>
  - <i><u>Only ROUGE-1 fell slightly, from 0.147 to 0.141.</u></i>
- <i><u>**The strict rubric moved the other way:**</u></i>
  - <i><u>Relevance fell from 3.09 to 2.95. It went up for 9 comments and down for 23, and 17 replies were now judged generic (score two), compared with 2 before.</u></i>
  - <i><u>Tone, factuality and clarity stayed the same.</u></i>
- <i><u>**Reliability.** The gain in BERTScore is small and could be chance: 57 of 100 improvements would happen about one time in five even if retrieval made no difference. The drop in relevance is clearer: 23 down against 9 up would happen by chance only about one time in fifty.</u></i>
- <i><u>**There were still no refusals.**</u></i>

<i><u>**Why the two kinds of measure disagree.** The retrieved expert replies are often short and general themselves; one human reference is simply "Having different cultures and races is good for societies in general." The model imitated that style. Its replies therefore became more similar to the references: the BLEU gain comes largely from copied expert phrases such as "There is no evidence that". But the replies engaged less with the specific claim. When the retrieved example contained a direct rebuttal, the reply improved. To "Most immigrants in the U.S. are criminals.", the model answered "There is no evidence that immigrants in the U.S. are more likely to be criminals than native-born citizens.", and relevance rose from three to four. When the examples were general, the reply became shorter and vaguer, for example "Addressing systemic issues can lead to better outcomes for everyone, including communities of color."</u></i>

<i><u>**Benefit and limitation.**</u></i>
- <i><u>**Benefit:** a small improvement in similarity to the expert replies. When a closely matching example contains a concrete rebuttal, the reply also becomes more direct.</u></i>
- <i><u>**Limitation:** the model copies the style of the examples, including their brevity and generality, so on average its replies engaged less with the specific claim. Higher overlap scores therefore do not mean better replies.</u></i>

<i><u>A possible next step would be to choose examples that rebut the claim directly, rather than only examples that are similar.</u></i>

### <i><u>Version C: full worked example with the real results and their interpretation</u></i>

<i><u>**Results from this run (100 test items):**</u></i>

| <i><u>Metric</u></i> | <i><u>Section 2: zero-shot</u></i> | <i><u>Section 3: retrieval few-shot</u></i> | <i><u>Difference</u></i> |
|---|---:|---:|---:|
| <i><u>BLEU</u></i> | <i><u>0.008</u></i> | <i><u>0.016</u></i> | <i><u>+0.008</u></i> |
| <i><u>ROUGE-1</u></i> | <i><u>0.147</u></i> | <i><u>0.141</u></i> | <i><u>−0.007</u></i> |
| <i><u>ROUGE-2</u></i> | <i><u>0.014</u></i> | <i><u>0.016</u></i> | <i><u>+0.002</u></i> |
| <i><u>ROUGE-L</u></i> | <i><u>0.108</u></i> | <i><u>0.110</u></i> | <i><u>+0.002</u></i> |
| <i><u>BERTScore-F1</u></i> | <i><u>0.254</u></i> | <i><u>0.262</u></i> | <i><u>+0.008</u></i> |
| <i><u>Refusal rate</u></i> | <i><u>0.000</u></i> | <i><u>0.000</u></i> | <i><u>0.000</u></i> |
| <i><u>Rubric relevance (1–5)</u></i> | <i><u>3.09</u></i> | <i><u>2.95</u></i> | <i><u>−0.14</u></i> |
| <i><u>Rubric tone (1–5)</u></i> | <i><u>4.04</u></i> | <i><u>4.05</u></i> | <i><u>+0.01</u></i> |
| <i><u>Rubric factuality (1–5)</u></i> | <i><u>4.95</u></i> | <i><u>4.95</u></i> | <i><u>0.00</u></i> |
| <i><u>Rubric clarity (1–5)</u></i> | <i><u>4.05</u></i> | <i><u>4.07</u></i> | <i><u>+0.02</u></i> |
| <i><u>Rubric overall (1–5)</u></i> | <i><u>4.03</u></i> | <i><u>4.01</u></i> | <i><u>−0.03</u></i> |

<i><u>**Relevance scores given (number of replies):**</u></i>

| <i><u>Relevance</u></i> | <i><u>Zero-shot</u></i> | <i><u>Retrieval</u></i> |
|---:|---:|---:|
| <i><u>2 (generic, ignores the claim)</u></i> | <i><u>2</u></i> | <i><u>17</u></i> |
| <i><u>3 (partly addresses the claim)</u></i> | <i><u>87</u></i> | <i><u>71</u></i> |
| <i><u>4 (addresses the claim, general argument)</u></i> | <i><u>11</u></i> | <i><u>12</u></i> |
| <i><u>5 (refutes with a concrete argument)</u></i> | <i><u>0</u></i> | <i><u>0</u></i> |

<i><u>Other results:</u></i>
- <i><u>BERTScore improved for 57 of 100 items.</u></i>
- <i><u>Relevance went up for 9 items and down for 23.</u></i>
- <i><u>The average similarity of the best retrieved example was 0.802.</u></i>
- <i><u>Judge answers that could not be read: 0.</u></i>

<i><u>**Real examples with the judge's verdicts (test items 1–3):**</u></i>

| <i><u>Hate speech</u></i> | <i><u>Zero-shot (relevance; critique)</u></i> | <i><u>Retrieval few-shot (relevance; critique)</u></i> |
|---|---|---|
| <i><u>"Most immigrants in the U.S. are criminals."</u></i> | <i><u>"Many immigrants contribute positively to society…" (**3**; "does not directly address the claim that immigrants are criminals")</u></i> | <i><u>"There is no evidence that immigrants in the U.S. are more likely to be criminals than native-born citizens." (**4**; "does not provide any concrete evidence to support its claim")</u></i> |
| <i><u>"Negroes create social problems so we must get rid of them."</u></i> | <i><u>"People of color face systemic challenges, but eradicating entire groups won't solve those issues…" (**3**; "focuses on systemic challenges")</u></i> | <i><u>"Addressing systemic issues can lead to better outcomes for everyone, including communities of color." (**3**; "does not directly address the specific claim")</u></i> |
| <i><u>"There is an invasion on a level never seen before in history…"</u></i> | <i><u>"This perspective overlooks the fact that many migrants seek better lives…" (**3**; "does not directly address the claim of cultural replacement")</u></i> | <i><u>"It's important to remember that migration happens all the time, and it doesn't necessarily mean an invasion…" (**3**; "does not directly refute the claim of an invasion")</u></i> |

<i><u>**Interpretation:**</u></i>
- <i><u>**Retrieval works as intended.** The best example is close to the test comment (average similarity 0.80; 0.925 for item 1).</u></i>
- <i><u>**A small improvement on the reference-based measures.** BLEU, ROUGE-2, ROUGE-L and BERTScore all rose: the replies became a little more similar to the expert replies. The gain is small, and it could be chance (57 of 100 BERTScore improvements: about one in five). The tutor confirmed that a small improvement is acceptable for the assignment.</u></i>
- <i><u>**The strict rubric shows a clear cost.** Relevance fell from 3.09 to 2.95, and the number of generic replies (score 2) rose from 2 to 17. With 23 replies worse and only 9 better, this is unlikely to be chance (about one in fifty).</u></i>
- <i><u>**Why both happen:** the model imitates the retrieved expert replies, and many of them are short and general. That raises the overlap with the references, which are written in the same style, but it reduces engagement with the specific claim.</u></i>
  - <i><u>The BLEU gain comes largely from copied expert phrases such as "There is no evidence that".</u></i>
- <i><u>**When the example contains a concrete rebuttal, retrieval helps.** Item 1 improves from relevance 3 to 4: the reply now contradicts the crime claim. The judge's critique also shows that the strict rubric still expects evidence for a 5.</u></i>
- <i><u>**When the examples are general, retrieval hurts.** Item 2 becomes shorter and vaguer.</u></i>
- <i><u>**Honest conclusion:**</u></i>
  - <i><u>Retrieval made the replies a little more like the expert replies, which is a small improvement on the reference-based metrics.</u></i>
  - <i><u>But it also made them engage less with the specific claim, as the strict rubric shows.</u></i>
  - <i><u>The main lesson: overlap with human references is not the same as quality (L7 p.45), and a rubric can reveal what the overlap metrics hide.</u></i>
  - <i><u>A better variant would select examples that contain a direct rebuttal (for example by reranking the retrieved examples, L7 p.21).</u></i>

<i><u>**Filled-in short answer:** Version A above already uses these real numbers.</u></i>

---
<i><u>**Study aid – not part of the core workbook.** Skip this part when you write your own submission.</u></i>

# <i><u>Exam revision summary: what this assignment practised</u></i>

| <i><u>Concept</u></i> | <i><u>Where it is used here</u></i> | <i><u>Course source</u></i> |
|---|---|---|
| <i><u>Tokens vs. types, tokenization</u></i> | <i><u>1a word counts</u></i> | <i><u>W2 p.8</u></i> |
| <i><u>Preprocessing choices (and when *not* to apply them)</u></i> | <i><u>Cleaning cell</u></i> | <i><u>L3 p.70</u></i> |
| <i><u>Data leakage, grouped and stratified train/test split</u></i> | <i><u>Split by unique hate speech</u></i> | <i><u>The brief; standard ML practice</u></i> |
| <i><u>Language model = next-token prediction; chain rule</u></i> | <i><u>Why decoder-only LLMs generate text</u></i> | <i><u>L5 p.12; L6 pp.13–16</u></i> |
| <i><u>Encoder-only vs. decoder-only vs. encoder–decoder</u></i> | <i><u>Choice of LLM</u></i> | <i><u>L3 p.91; L6 p.19</u></i> |
| <i><u>Instruction tuning, chat templates (hidden tags)</u></i> | <i><u>Model choice, `apply_chat_template`</u></i> | <i><u>L6 p.43</u></i> |
| <i><u>Quantisation (4-bit / 8-bit) to fit models in GPU memory</u></i> | <i><u>`BitsAndBytesConfig`</u></i> | <i><u>Lab7 (4-bit), WS6b (8-bit)</u></i> |
| <i><u>Pre-train, prompt, predict (no weight updates)</u></i> | <i><u>The whole assignment (inference only)</u></i> | <i><u>L6 pp.40–41</u></i> |
| <i><u>Prompt elements; AUTOMAT; CO-STAR; positive instructions; fail cases</u></i> | <i><u>Prompt design</u></i> | <i><u>L6 pp.46–51, 58, 65–69</u></i> |
| <i><u>Output formatting (JSON for downstream processing)</u></i> | <i><u>Judge answers</u></i> | <i><u>L6 p.58</u></i> |
| <i><u>Zero-shot vs. few-shot vs. chain-of-thought</u></i> | <i><u>Section 2 vs. Section 3; alternatives</u></i> | <i><u>L6 pp.53–56; L3 p.93; Lab7</u></i> |
| <i><u>Greedy vs. sampling vs. temperature; softmax</u></i> | <i><u>Decoding choice</u></i> | <i><u>L7 pp.47–50; L6 p.17; WS7b</u></i> |
| <i><u>Hallucination (intrinsic / extrinsic) and its causes</u></i> | <i><u>"Fact-bound" rules, factuality criterion</u></i> | <i><u>L7 pp.43–44</u></i> |
| <i><u>BLEU / ROUGE (overlap, precision / recall), F1</u></i> | <i><u>2b metrics</u></i> | <i><u>Lab7; L4 pp.31–32; L7 p.45</u></i> |
| <i><u>Cosine similarity, embeddings, BERT</u></i> | <i><u>BERTScore, retrieval</u></i> | <i><u>L3 pp.30–32, 85</u></i> |
| <i><u>Evaluation dimensions, LLM as judge; judge leniency (ceiling effect) and critique-first scoring</u></i> | <i><u>Rubric</u></i> | <i><u>L7 pp.27, 41; L6 p.55</u></i> |
| <i><u>RAG: Retrieve-Read, dense vs. sparse retrieval, Sentence Transformers</u></i> | <i><u>Section 3</u></i> | <i><u>L7 pp.5, 11–14</u></i> |
| <i><u>Reranking ("lost in the middle"), filtering for diversity, MMR</u></i> | <i><u>Example order, duplicate filter</u></i> | <i><u>L7 pp.21–23</u></i> |
| <i><u>RAG vs. fine-tuning</u></i> | <i><u>Why RAG fits "inference only"</u></i> | <i><u>L7 p.29</u></i> |

---
# That's a wrap!

Before submitting your own version:
- run all cells top to bottom with saved outputs;
- write the 50-word answers in your own words.